# MVP: Pipeline de Dados na Nuvem — IMDB Top 1000

**Aluno:** Gabriel Simas Gomes Barata  
**Plataforma:** Databricks Free Edition  
**Arquitetura:** Medalhão (Bronze → Silver → Gold) sobre Delta Lake  
**Dataset:** IMDB Dataset of Top 1000 Movies and TV Shows — licença **CC0: Public Domain**  
**Repositório GitHub:** https://github.com/gabrielbarata/MVP-engenharia-de-dados  
**Notebook:** `MVP.ipynb`

## 0. Setup — Databricks Repos e Reprodutibilidade

Todo o código deste MVP está versionado em repositório público do GitHub e sincronizado
com o Databricks via **Databricks Repos**, atendendo ao requisito de entrega.

**Repositório:** https://github.com/gabrielbarata/MVP-engenharia-de-dados

**Como reproduzir:**
1. No Databricks Free Edition: **Workspace → Repos → Add Repo**
2. URL do repositório: `https://github.com/gabrielbarata/MVP-engenharia-de-dados`
3. Clonar e abrir `MVP.ipynb` a partir do Repo (não do Workspace local — assim o versionamento fica ativo)
4. Executar as células na ordem

**Referência:** [Getting Started with Databricks Repos](https://docs.databricks.com/aws/en/repos/)

> **Evidência:** `docs/screenshots/00_repos.png` — screenshot do Repo clonado com o notebook listado.

## 1. Contexto de Negócios e Perguntas (Etapas 2 e 4.1)

### Problema
Streaming services, produtoras e investidores precisam decidir **que tipo de filme produzir, comprar ou recomendar**. Errar essa decisão custa milhões. Este MVP constrói um pipeline que organiza dados do IMDB Top 1000 para responder, com base em evidências, **o que caracteriza um filme bem avaliado**.

### Perguntas de Negócio
1. **PG1** — Quais **gêneros** possuem a maior média de avaliação IMDB?
2. **PG2** — Filmes lançados **a partir de 2000** têm avaliação média superior aos anteriores?
3. **PG3** — A **duração** do filme influencia a nota IMDB?
4. **PG4** — Existe **correlação entre número de votos e nota** IMDB?
5. **PG5** — Quais **diretores** (com ≥ 3 filmes no Top 1000) têm as melhores médias?

### Licença dos dados
Dataset público disponibilizado no Kaggle por *Harshit Shankhdhar* sob licença **CC0: Public Domain** — uso livre, inclusive comercial, sem necessidade de atribuição. Fonte original: IMDB.

In [0]:
%pip install kagglehub -qq
import kagglehub

# Download latest version
path = kagglehub.dataset_download("harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows")

print("Path to dataset files:", path)

In [0]:
import os
print(os.listdir(path))

In [0]:
import pandas as pd

file_path = os.path.join(path, "imdb_top_1000.csv")
df = pd.read_csv(file_path)
print(f"Shape: {df.shape}")
df.head()

## 2. Carga dos Dados — Camada Bronze (Etapa 4.2)

Os dados brutos são lidos via **kagglehub** (API oficial do Kaggle) diretamente no cluster Databricks e convertidos para Spark DataFrame **sem qualquer transformação semântica**. Adiciona-se apenas metadado de controle: `ingested_at` (timestamp da ingestão) e `source_file` (origem).

Regra da camada Bronze: **preservar o dado como veio**, garantindo rastreabilidade e reprocessamento.

In [0]:
from pyspark.sql.functions import current_timestamp, lit

# Converter para Spark DataFrame
df_bronze = spark.createDataFrame(df)

# Adicionar metadados de controle (rastreabilidade)
df_bronze = (df_bronze
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_file", lit("imdb_top_1000.csv"))
)

df_bronze.printSchema()
display(df_bronze.limit(5))

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;
SHOW SCHEMAS; 

In [0]:
(df_bronze.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable("bronze.imdb_raw"))

In [0]:
%sql
SELECT * FROM bronze.imdb_raw LIMIT 10;

## 3. Camada Silver — Limpeza e Padronização

Transformações aplicadas:
- Remoção de duplicatas exatas
- `Released_Year`: convertido para inteiro, aceitando apenas valores numéricos válidos
- `IMDB_Rating`: garantido como float no intervalo [0,10]
- `Runtime_min`: extraído o número de "142 min" → 142
- `Meta_score`: cast para inteiro (nulos mantidos, pois ausência é informativa)
- `Gross`: remoção de vírgulas e cast para double
- Filtro: mantém apenas linhas com `Released_Year` e `IMDB_Rating` não nulos (essenciais para as análises)

In [0]:
from pyspark.sql.functions import (
    col, when, regexp_extract, regexp_replace, trim
)

df_silver = (df_bronze
    .dropDuplicates()
    # Released_Year: string → int (só se for numérico)
    .withColumn("Released_Year",
        when(col("Released_Year").rlike("^[0-9]{4}$"), col("Released_Year"))
        .otherwise(None).cast("int"))
    # IMDB_Rating: garantir float
    .withColumn("IMDB_Rating", col("IMDB_Rating").cast("float"))
    # Runtime: "142 min" → 142
    .withColumn("Runtime_min",
        regexp_extract(col("Runtime"), r"(\d+)", 1).cast("int"))
    # Meta_score: double → int
    .withColumn("Meta_score", col("Meta_score").cast("int"))
    # No_of_Votes: garantir long
    .withColumn("No_of_Votes", col("No_of_Votes").cast("long"))
    # Gross: "28,341,469" → 28341469.0
    .withColumn("Gross",
        regexp_replace(col("Gross"), ",", "").cast("double"))
    # Trim em colunas de texto (consistência)
    .withColumn("Certificate", trim(col("Certificate")))
    .withColumn("Genre", trim(col("Genre")))
    .withColumn("Director", trim(col("Director")))
    # Filtro de linhas essenciais
    .filter(col("Released_Year").isNotNull() & col("IMDB_Rating").isNotNull())
)

print(f"Linhas após limpeza: {df_silver.count()}")
df_silver.printSchema()

In [0]:
(df_silver.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable("silver.imdb_clean"))

In [0]:
%sql
SELECT * FROM silver.imdb_clean LIMIT 10;

## 4. Camada Gold — Modelagem para Responder as Perguntas

Modelo híbrido: **tabela fato** (`fato_filmes`) + **agregações especializadas** por pergunta + **dimensão de gênero explodida** (`dim_genero`) para permitir análises por gênero individual.

Justificativa: o campo `Genre` no IMDB é multivalorado ("Drama, Crime"). Explodir em uma tabela dedicada é a forma correta de modelar (evita dupla contagem em médias).

In [0]:
from pyspark.sql.functions import explode, split, trim

df_genero = (df_silver
    .select("Series_Title", "IMDB_Rating", "Released_Year",
            explode(split(col("Genre"), ",")).alias("genero"))
    .withColumn("genero", trim(col("genero")))
)

(df_genero.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable("gold.dim_genero"))

display(df_genero.limit(10))

In [0]:
%sql
CREATE OR REPLACE TABLE gold.fato_filmes AS
SELECT
  Series_Title,
  Director,
  Certificate,
  Released_Year,
  Runtime_min,
  Genre,
  IMDB_Rating,
  Meta_score,
  No_of_Votes,
  Gross,
  CASE WHEN Released_Year >= 2000 THEN '2000+' ELSE 'Antes de 2000' END AS periodo,
  CASE
    WHEN Runtime_min < 100 THEN 'Curto (<100min)'
    WHEN Runtime_min <= 150 THEN 'Médio (100-150min)'
    ELSE 'Longo (>150min)'
  END AS faixa_duracao
FROM silver.imdb_clean;
SELECT * FROM gold.fato_filmes LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE gold.avg_rating_by_genre AS
SELECT
  genero,
  ROUND(AVG(IMDB_Rating), 3) AS media_nota,
  COUNT(*) AS qtd_filmes
FROM gold.dim_genero
GROUP BY genero
HAVING COUNT(*) >= 5
ORDER BY media_nota DESC;
SELECT * FROM gold.avg_rating_by_genre LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE gold.avg_rating_by_decade AS
SELECT
  (Released_Year DIV 10) * 10 AS decada,
  ROUND(AVG(IMDB_Rating), 3) AS media_nota,
  COUNT(*) AS qtd_filmes
FROM silver.imdb_clean
GROUP BY decada
ORDER BY decada;
SELECT * FROM gold.avg_rating_by_decade LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE gold.avg_rating_by_duration AS
SELECT
  faixa_duracao,
  ROUND(AVG(IMDB_Rating), 3) AS media_nota,
  ROUND(AVG(No_of_Votes), 0) AS media_votos,
  COUNT(*) AS qtd_filmes
FROM gold.fato_filmes
GROUP BY faixa_duracao
ORDER BY media_nota DESC;
SELECT * FROM gold.avg_rating_by_duration LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE gold.top_directors AS
SELECT
  Director,
  ROUND(AVG(IMDB_Rating), 3) AS media_nota,
  COUNT(*) AS qtd_filmes
FROM silver.imdb_clean
GROUP BY Director
HAVING COUNT(*) >= 3
ORDER BY media_nota DESC;
SELECT * FROM gold.top_directors LIMIT 10;

## 5. Qualidade de Dados (Etapa 4.5)

Verificamos 6 dimensões clássicas: **completude, unicidade, consistência, acurácia, domínio e outliers**. Cada checagem é documentada com o resultado obtido.

In [0]:
from pyspark.sql.functions import col, count, when, mean, stddev, isnan

df_qualidade = spark.read.table("silver.imdb_clean")

print("=" * 60)
print("=== 1. COMPLETUDE: Nulos por coluna ===")
print("=" * 60)
df_qualidade.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in df_qualidade.columns]
).show()

print("=" * 60)
print("=== 2. UNICIDADE ===")
print("=" * 60)
total = df_qualidade.count()
distintos = df_qualidade.distinct().count()
print(f"Total de registros: {total}")
print(f"Registros distintos: {distintos}")
print(f"Duplicatas: {total - distintos}")

print("=" * 60)
print("=== 3. ESTATÍSTICAS (colunas numéricas) ===")
print("=" * 60)
numericas = [c for c, t in df_qualidade.dtypes if t in ("int", "double", "float", "long")]
df_qualidade.select(numericas).describe().show()

print("=" * 60)
print("=== 4. ACURÁCIA / DOMÍNIO ===")
print("=" * 60)

n_nota_fora = df_qualidade.filter(
    (col("IMDB_Rating") < 0) | (col("IMDB_Rating") > 10)
).count()
print(f"Notas fora de [0,10]: {n_nota_fora}")

n_ano_fora = df_qualidade.filter(
    (col("Released_Year") < 1900) | (col("Released_Year") > 2025)
).count()
print(f"Anos fora de [1900,2025]: {n_ano_fora}")

n_duracao_fora = df_qualidade.filter(
    (col("Runtime_min") < 1) | (col("Runtime_min") > 1000)
).count()
print(f"Durações fora de [1,1000] min: {n_duracao_fora}")

print("=" * 60)
print("=== 5. CONSISTÊNCIA ===")
print("=" * 60)
print("Top 10 valores de Certificate:")
df_qualidade.groupBy("Certificate").count().orderBy("count", ascending=False).show(10)

print("Top 10 combinações de Genre (multivalorado):")
df_qualidade.groupBy("Genre").count().orderBy("count", ascending=False).show(10)

print("=" * 60)
print("=== 6. OUTLIERS — regra 2σ (bilateral) ===")
print("=" * 60)
m = df_qualidade.select(mean("IMDB_Rating")).first()[0]
s = df_qualidade.select(stddev("IMDB_Rating")).first()[0]
lim_inf = m - 2 * s
lim_sup = m + 2 * s

print(f"Média = {m:.2f} | Desvio padrão = {s:.2f}")
print(f"Limite inferior (μ - 2σ) = {lim_inf:.2f}")
print(f"Limite superior (μ + 2σ) = {lim_sup:.2f}")

print("\n--- Outliers INFERIORES (nota < μ - 2σ) ---")
df_qualidade.filter(col("IMDB_Rating") < lim_inf) \
    .select("Series_Title", "IMDB_Rating") \
    .orderBy("IMDB_Rating") \
    .show(truncate=False)

print("\n--- Outliers SUPERIORES (nota > μ + 2σ) ---")
df_qualidade.filter(col("IMDB_Rating") > lim_sup) \
    .select("Series_Title", "IMDB_Rating") \
    .orderBy(col("IMDB_Rating").desc()) \
    .show(truncate=False)

n_inf = df_qualidade.filter(col("IMDB_Rating") < lim_inf).count()
n_sup = df_qualidade.filter(col("IMDB_Rating") > lim_sup).count()
print("\n--- Resumo 2σ ---")
print(f"Total de filmes: {total}")
print(f"Outliers inferiores: {n_inf} ({n_inf/total*100:.2f}%)")
print(f"Outliers superiores: {n_sup} ({n_sup/total*100:.2f}%)")

print("=" * 60)
print("=== 6b. OUTLIERS — regra IQR (mais robusta) ===")
print("=" * 60)
q1, q3 = df_qualidade.approxQuantile("IMDB_Rating", [0.25, 0.75], 0.0)
iqr = q3 - q1
lim_inf_iqr = q1 - 1.5 * iqr
lim_sup_iqr = q3 + 1.5 * iqr

print(f"Q1 = {q1:.2f} | Q3 = {q3:.2f} | IQR = {iqr:.2f}")
print(f"Limite inferior (Q1 - 1.5×IQR) = {lim_inf_iqr:.2f}")
print(f"Limite superior (Q3 + 1.5×IQR) = {lim_sup_iqr:.2f}")

print("\n--- Outliers INFERIORES (IQR) ---")
df_qualidade.filter(col("IMDB_Rating") < lim_inf_iqr) \
    .select("Series_Title", "IMDB_Rating") \
    .orderBy("IMDB_Rating") \
    .show(truncate=False)

print("\n--- Outliers SUPERIORES (IQR) ---")
df_qualidade.filter(col("IMDB_Rating") > lim_sup_iqr) \
    .select("Series_Title", "IMDB_Rating") \
    .orderBy(col("IMDB_Rating").desc()) \
    .show(truncate=False)

n_inf_iqr = df_qualidade.filter(col("IMDB_Rating") < lim_inf_iqr).count()
n_sup_iqr = df_qualidade.filter(col("IMDB_Rating") > lim_sup_iqr).count()
print("\n--- Resumo IQR ---")
print(f"Outliers inferiores (IQR): {n_inf_iqr} ({n_inf_iqr/total*100:.2f}%)")
print(f"Outliers superiores (IQR): {n_sup_iqr} ({n_sup_iqr/total*100:.2f}%)")

print("\n" + "=" * 60)
print("FIM DOS CHECKS DE QUALIDADE")
print("=" * 60)

In [0]:
# === Persistir resumo de qualidade em gold.dq_summary ===
from pyspark.sql.functions import col, mean, stddev

df_q = spark.read.table("silver.imdb_clean")
total = df_q.count()
distintos = df_q.distinct().count()

# --- Completude (nulos por coluna essencial) ---
nulos_title    = df_q.filter(col("Series_Title").isNull()).count()
nulos_director = df_q.filter(col("Director").isNull()).count()
nulos_overview = df_q.filter(col("Overview").isNull()).count()
nulos_cert     = df_q.filter(col("Certificate").isNull()).count()
nulos_meta     = df_q.filter(col("Meta_score").isNull()).count()
nulos_gross    = df_q.filter(col("Gross").isNull()).count()

# --- Acurácia / domínio ---
notas_fora = df_q.filter((col("IMDB_Rating") < 0) | (col("IMDB_Rating") > 10)).count()
anos_fora  = df_q.filter((col("Released_Year") < 1900) | (col("Released_Year") > 2025)).count()
dur_fora   = df_q.filter((col("Runtime_min") < 1) | (col("Runtime_min") > 1000)).count()

# --- Outliers (regra 2σ bilateral) ---
m = df_q.select(mean("IMDB_Rating")).first()[0]
s = df_q.select(stddev("IMDB_Rating")).first()[0]
out_inf = df_q.filter(col("IMDB_Rating") < (m - 2*s)).count()
out_sup = df_q.filter(col("IMDB_Rating") > (m + 2*s)).count()

dq_data = [
    # Unicidade
    ("total_registros",      float(total),           "Registros após limpeza"),
    ("registros_distintos",  float(distintos),       "Registros únicos"),
    ("duplicatas",           float(total - distintos), "Duplicatas detectadas"),
    # Completude — campos essenciais
    ("nulos_series_title",   float(nulos_title),     "Título ausente"),
    ("nulos_director",       float(nulos_director),  "Diretor ausente"),
    ("nulos_overview",       float(nulos_overview),  "Sinopse ausente"),
    # Completude — campos opcionais
    ("nulos_certificate",    float(nulos_cert),      "Classificação etária ausente (informativo)"),
    ("nulos_meta_score",     float(nulos_meta),      "Metacritic ausente (informativo)"),
    ("nulos_gross",          float(nulos_gross),     "Bilheteria ausente (informativo)"),
    # Acurácia / domínio
    ("notas_fora_0_10",      float(notas_fora),      "Notas IMDB fora de [0,10]"),
    ("anos_fora_1900_2025",  float(anos_fora),       "Anos fora de [1900,2025]"),
    ("duracoes_fora_1_1000", float(dur_fora),        "Durações fora de [1,1000] min"),
    # Outliers 2σ
    ("outliers_2sigma_inf",  float(out_inf),         "Nota < μ-2σ (bilateral)"),
    ("outliers_2sigma_sup",  float(out_sup),         "Nota > μ+2σ (bilateral)"),
]

dq_df = spark.createDataFrame(dq_data, ["metrica", "valor", "detalhe"])

(dq_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable("gold.dq_summary"))

# Documentar a tabela no Unity Catalog
spark.sql("""
    COMMENT ON TABLE gold.dq_summary IS
    'Resumo auditável dos indicadores de qualidade de dados (completude, unicidade, acurácia, domínio e outliers). Cada linha = 1 métrica com nome, valor medido e contexto.'
""")

display(dq_df)

### 5.1 Persistência do resumo de qualidade

Os indicadores calculados foram consolidados na tabela Delta `gold.dq_summary`,
que serve como **artefato auditável** de qualidade. Cada linha representa uma
métrica com nome, valor medido e descrição contextual.

**Colunas:** `metrica` (string), `valor` (double), `detalhe` (string).

**Evidência:** ver `docs/screenshots/11_dq_summary.png`.

In [0]:
%sql
-- === Documentação de COLUNAS no Unity Catalog (Catálogo de Dados) ===
-- Completa os COMMENT ON TABLE já existentes.

-- BRONZE
ALTER TABLE bronze.imdb_raw ALTER COLUMN Series_Title    COMMENT 'Título do filme/série (texto livre).';
ALTER TABLE bronze.imdb_raw ALTER COLUMN Released_Year   COMMENT 'Ano de lançamento em formato texto (4 dígitos).';
ALTER TABLE bronze.imdb_raw ALTER COLUMN Certificate     COMMENT 'Classificação etária: G, PG, PG-13, R, U, A, UA, Passed, Approved.';
ALTER TABLE bronze.imdb_raw ALTER COLUMN Runtime         COMMENT 'Duração bruta no formato "142 min".';
ALTER TABLE bronze.imdb_raw ALTER COLUMN Genre           COMMENT 'Gêneros separados por vírgula (multivalorado).';
ALTER TABLE bronze.imdb_raw ALTER COLUMN IMDB_Rating     COMMENT 'Nota IMDB. Domínio: [0,10].';
ALTER TABLE bronze.imdb_raw ALTER COLUMN Meta_score      COMMENT 'Nota Metacritic. Domínio: [0,100]. Nulo quando indisponível.';
ALTER TABLE bronze.imdb_raw ALTER COLUMN Director        COMMENT 'Nome do diretor (texto livre).';
ALTER TABLE bronze.imdb_raw ALTER COLUMN No_of_Votes     COMMENT 'Número total de votos no IMDB. > 0.';
ALTER TABLE bronze.imdb_raw ALTER COLUMN Gross           COMMENT 'Bilheteria bruta em USD no formato "28,341,469".';
ALTER TABLE bronze.imdb_raw ALTER COLUMN ingested_at     COMMENT 'Timestamp UTC da ingestão (metadado de controle).';
ALTER TABLE bronze.imdb_raw ALTER COLUMN source_file     COMMENT 'Arquivo CSV de origem (rastreabilidade).';

-- SILVER
ALTER TABLE silver.imdb_clean ALTER COLUMN Released_Year COMMENT 'Ano convertido para inteiro. Filtro: apenas valores em [1900,2025].';
ALTER TABLE silver.imdb_clean ALTER COLUMN Runtime_min   COMMENT 'Duração em minutos (derivada de Runtime via regexp_extract).';
ALTER TABLE silver.imdb_clean ALTER COLUMN Meta_score    COMMENT 'Metacritic convertido para inteiro. Nulos mantidos (ausência informativa).';
ALTER TABLE silver.imdb_clean ALTER COLUMN Gross         COMMENT 'Bilheteria em USD (double, vírgulas removidas).';

-- GOLD — fato_filmes
ALTER TABLE gold.fato_filmes ALTER COLUMN periodo        COMMENT 'Faixa temporal: "2000+" se Released_Year >= 2000, senão "Antes de 2000".';
ALTER TABLE gold.fato_filmes ALTER COLUMN faixa_duracao  COMMENT 'Faixa de duração: Curto (<100min), Médio (100-150min), Longo (>150min).';

-- GOLD — dim_genero
ALTER TABLE gold.dim_genero ALTER COLUMN genero          COMMENT 'Gênero individual (explode de Genre multivalorado).';

In [0]:
%sql
COMMENT ON TABLE bronze.imdb_raw IS 'Dados brutos do IMDB Top 1000 (Kaggle CC0). Preserva formato original + metadados de ingestão.';
COMMENT ON TABLE silver.imdb_clean IS 'Dados limpos: tipos corrigidos, Runtime_min derivado, deduplicado, filtrado para Released_Year e IMDB_Rating não nulos.';
COMMENT ON TABLE gold.fato_filmes IS 'Fato central de filmes com faixas derivadas (periodo, faixa_duracao). Fonte para análises de negócio.';
COMMENT ON TABLE gold.dim_genero IS 'Dimensão de gênero explodida (1 linha por filme-gênero). Evita dupla contagem em médias.';
COMMENT ON TABLE gold.dq_summary IS 'Resumo auditável dos indicadores de qualidade de dados (completude, unicidade, acurácia, outliers).';

In [0]:
# === EVIDÊNCIA: tabelas persistidas por schema ===
for schema in ["bronze", "silver", "gold"]:
    print(f"\n{'='*60}\nTabelas no schema: {schema}\n{'='*60}")
    spark.sql(f"SHOW TABLES IN {schema}").show(truncate=False)

## 6. Análise Final — Respostas às Perguntas de Negócio

In [0]:
%sql
-- PG1: Quais gêneros possuem a maior média de avaliação?
SELECT * FROM gold.avg_rating_by_genre LIMIT 10;

### 📊 Discussão — PG1: Quais gêneros têm a maior média?

**Resultado observado:**
| Gênero | Média | Nº filmes |
|--------|-------|-----------|
| War | 8.014 | 51 |
| Western | 8.000 | 20 |
| Film-Noir | 7.989 | 19 |
| Sci-Fi | 7.978 | 67 |
| Mystery | 7.968 | 99 |
| Drama | 7.960 | **723** |

**Interpretação:** Gêneros de nicho (War, Western, Film-Noir) lideram a média, mas com volume modesto (19–51 filmes). O efeito é clássico de **survivorship bias**: só entram no Top 1000 as obras mais aclamadas desses gêneros. Já **Drama**, com 723 filmes (o gênero dominante), mantém média competitiva (7.96) mesmo com amostra 14× maior — sinal de robustez.

**Recomendação de negócio:** para uma plataforma de streaming, produzir ou licenciar **Drama** entrega o melhor equilíbrio entre qualidade média e apelo de catálogo. Gêneros de nicho funcionam como *prestígio* pontual, não como estratégia de volume.

In [0]:
%sql
-- PG2b (complemento): visão por década — revela efeito de amostra
SELECT
  decada,
  media_nota,
  qtd_filmes,
  ROUND(media_nota * qtd_filmes, 1) AS peso_na_media_geral
FROM gold.avg_rating_by_decade
ORDER BY decada;

In [0]:
%sql
-- PG2: Filmes a partir de 2000 têm nota superior?
SELECT
  periodo,
  ROUND(AVG(IMDB_Rating), 3) AS media_nota,
  COUNT(*) AS qtd
FROM gold.fato_filmes
GROUP BY periodo;

### 📊 Discussão — PG2: Filmes pós-2000 são mais bem avaliados?

**Resultado observado:**
| Período | Média | Nº filmes |
|---------|-------|-----------|
| Antes de 2000 | **7.982** | 514 |
| 2000+ | 7.915 | 485 |

**Interpretação:** Ao contrário do senso comum de que "cinema moderno é melhor", os filmes **anteriores a 2000 têm média ligeiramente superior** (+0.067). A diferença é pequena, mas consistente — e reflete o *viés de curadoria* do próprio dataset: o Top 1000 inclui obras históricas que passaram pelo teste do tempo, enquanto o cinema recente ainda está sendo avaliado por um público menos consolidado.

**Recomendação de negócio:** a era de lançamento **não é preditor forte** de qualidade. Investir em catálogos vintage tem apelo comprovado; investir em lançamentos recentes exige critérios adicionais (diretor, gênero, elenco).

### 📊 Discussão complementar — PG2b: Efeito de década

A visão por década mostra que a aparente vantagem dos filmes pré-2000 **não é uniforme**:
as décadas de 1920–1950 têm médias altas (8.0–8.13) mas amostras pequenas (11–56 filmes),
enquanto 2000 e 2010 somam **479 filmes** com médias próximas de 7.90–7.92.

O que parece "cinema antigo é melhor" é, na prática, **efeito de amostra + viés de
curadoria**: poucos títulos antigos chegaram ao Top 1000, e esses poucos foram
selecionados por excelência. Quando a amostra cresce (pós-2000), a média se aproxima
do valor real do conjunto. Por isso, a era de lançamento **não deve ser usada como
preditor isolado** de qualidade.

**Detalhe metodológico:** a década de 2020 aparece com média 8.133, mas com
apenas **6 filmes** — número estatisticamente irrelevante (uma única obra-prima
distorce a média inteira). Esse é o mesmo efeito de amostra pequena que
explica as décadas de 1920 e 1950. Reforça a conclusão: **análise por era só
é confiável quando há volume amostral suficiente** (n ≥ 30, idealmente).

In [0]:
%sql
-- PG3: Duração influencia a nota?
SELECT * FROM gold.avg_rating_by_duration;

### 📊 Discussão — PG3: Duração influencia a nota?

**Resultado observado:**
| Faixa | Média nota | Média votos | Nº filmes |
|-------|-----------|-------------|-----------|
| Longo (>150min) | **8.097** | 362.546 | 147 |
| Médio (100–150min) | 7.928 | 281.390 | 667 |
| Curto (<100min) | 7.910 | 175.363 | 185 |

**Interpretação:** Filmes **longos têm média significativamente maior** (+0.17 vs. médio), e também atraem mais votos. Existe uma correlação implícita entre duração, ambição artística e engajamento do público — épicos, dramas históricos e obras de prestígio costumam ser longos. Mas **causa e efeito devem ser separados**: não é "ser longo" que gera boa nota, é o tipo de filme (ambicioso, prestigiado) que tende a ser longo.

**Recomendação de negócio:** para produção original de prestígio, duração de 150+ min tem correlação histórica com performance crítica — mas o custo de produção também sobe. O "sweet spot comercial" continua sendo 100–150 min.

In [0]:
%sql
-- PG4a: Média de nota por faixa de votos
SELECT
  CASE
    WHEN No_of_Votes < 100000 THEN 'Baixo (<100k)'
    WHEN No_of_Votes < 500000 THEN 'Médio (100k-500k)'
    ELSE 'Alto (500k+)'
  END AS faixa_votos,
  ROUND(AVG(IMDB_Rating), 3) AS media_nota,
  COUNT(*) AS qtd
FROM gold.fato_filmes
GROUP BY faixa_votos
ORDER BY media_nota DESC;

In [0]:
%sql
-- PG4b: Coeficiente de correlação (Pearson)
SELECT ROUND(corr(No_of_Votes, IMDB_Rating), 4) AS correlacao
FROM gold.fato_filmes;

### 📊 Discussão — PG4: Existe correlação entre votos e nota?

**Resultado observado:** Coeficiente de Pearson **r = 0.4954** (correlação positiva moderada).

**Interpretação:** r ≈ 0.50 significa que **popularidade e qualidade andam juntas, mas não perfeitamente**. Cerca de 25% da variação da nota é explicada pelo número de votos (r² ≈ 0.245). Isso é esperado: filmes mais votados passam por um crivo maior de público e a média tende a estabilizar em valores "consensuais". Filmes com poucos votos são mais voláteis — uma avaliação extrema pesa mais.

**Recomendação de negócio:** ao usar popularidade como proxy de qualidade, tenha cuidado. Existem **joias escondidas** (alta nota, baixo volume) — ótimas para curadoria editorial e para estratégias de "descoberta" que diferenciam a plataforma. A correlação moderada também indica que **marketing sozinho não garante boa nota**.

In [0]:
%sql
-- PG5: Melhores diretores (≥3 filmes)
SELECT * FROM gold.top_directors LIMIT 10;

### 📊 Discussão — PG5: Quais diretores têm as melhores médias?

**Resultado observado (top 10, ≥ 3 filmes no Top 1000):**
| Diretor | Média | Nº filmes |
|---------|-------|-----------|
| Christopher Nolan | **8.462** | 8 |
| Peter Jackson | 8.400 | 5 |
| Francis Ford Coppola | 8.400 | 5 |
| Charles Chaplin | 8.333 | 6 |
| Sergio Leone | 8.267 | 6 |
| Stanley Kubrick | 8.233 | 9 |
| Akira Kurosawa | 8.220 | 10 |

**Interpretação:** Christopher Nolan se destaca não só pela **maior média (8.46)** mas por **consistência em 8 filmes** — o que é notável em uma indústria onde o acerto médio é baixo. Nomes como Kubrick (9 filmes) e Kurosawa (10 filmes) mantêm médias acima de 8.2 mesmo com volume alto, o que é ainda mais impressionante do que liderar com poucos títulos.

**Recomendação de negócio:** para aquisição de catálogo ou contratos de produção, diretores com **histórico recorrente e alta média** reduzem o risco percebido. Nolan, Jackson, Coppola e Kubrick formam o "núcleo duro" de menor risco/retorno consistente.

## 6.1 Visualizações Complementares

Gráficos gerados a partir das tabelas Gold para reforçar a leitura dos resultados
das 5 perguntas de negócio.

In [0]:
# === VISUALIZAÇÕES (evidências gráficas) ===
import matplotlib.pyplot as plt

# PG1 — Top 10 gêneros
df1 = spark.sql("SELECT genero, media_nota FROM gold.avg_rating_by_genre ORDER BY media_nota DESC LIMIT 10").toPandas()
plt.figure(figsize=(10,5))
plt.barh(df1["genero"], df1["media_nota"], color="steelblue")
plt.xlabel("Média IMDB"); plt.title("PG1 — Top 10 gêneros por média IMDB")
plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

# PG2 — Antes vs Depois de 2000
df2 = spark.sql("""
    SELECT periodo, ROUND(AVG(IMDB_Rating),3) AS media_nota
    FROM gold.fato_filmes GROUP BY periodo ORDER BY periodo
""").toPandas()
plt.figure(figsize=(6,5))
plt.bar(df2["periodo"], df2["media_nota"], color="mediumpurple")
plt.ylabel("Média IMDB"); plt.ylim(7.5, 8.1)
plt.title("PG2 — Média por período (Antes vs Depois de 2000)")
plt.tight_layout(); plt.show()

# PG3 — Duração
df3 = spark.sql("SELECT faixa_duracao, media_nota FROM gold.avg_rating_by_duration ORDER BY media_nota DESC").toPandas()
plt.figure(figsize=(7,5))
plt.bar(df3["faixa_duracao"], df3["media_nota"], color="darkorange")
plt.ylabel("Média IMDB"); plt.title("PG3 — Média por faixa de duração")
plt.tight_layout(); plt.show()

# PG4 — Correlação votos × nota (scatter)
df4 = spark.sql("SELECT No_of_Votes, IMDB_Rating FROM gold.fato_filmes").toPandas()
plt.figure(figsize=(9,5))
plt.scatter(df4["No_of_Votes"], df4["IMDB_Rating"], alpha=0.4, color="crimson")
plt.xlabel("Nº de votos"); plt.ylabel("Nota IMDB")
plt.title("PG4 — Correlação votos × nota (Pearson r = 0.4954)")
plt.xscale("log"); plt.tight_layout(); plt.show()

# PG5 — Top diretores
df5 = spark.sql("SELECT Director, media_nota FROM gold.top_directors LIMIT 10").toPandas()
plt.figure(figsize=(10,5))
plt.barh(df5["Director"], df5["media_nota"], color="seagreen")
plt.xlabel("Média IMDB"); plt.title("PG5 — Top 10 diretores (≥3 filmes)")
plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

### 🎯 Conclusão Geral — o que caracteriza um filme bem avaliado?

Cruzando as 5 análises, o **perfil de um filme com alta nota IMDB no Top 1000** é:

1. **Gênero:** Drama (combinado com Crime), ou nichos de prestígio (War, Film-Noir) quando o objetivo é média máxima.
2. **Duração:** acima de 150 min — épicos e dramas históricos dominam o topo.
3. **Popularidade:** alta (r = 0.50 com a nota), mas há espaço para joias de nicho.
4. **Direção:** nomes com histórico consistente (Nolan, Kubrick, Kurosawa).
5. **Era:** fator secundário — filmes pré-2000 têm ligeira vantagem, mas não é decisivo.

**Aplicação prática para streaming/produtora:** priorizar **Drama de longa duração com diretor de histórico comprovado** maximiza a probabilidade de nota alta; usar **popularidade como sinal parcial** (não como critério único); e explorar **joias escondidas** em nichos para diferenciar o catálogo.

**Limitações reconhecidas:** o dataset é um recorte curado (Top 1000), não uma amostra aleatória do mercado cinematográfico. Conclusões não se generalizam para "qualquer filme lançado" — apenas para filmes que já atingiram o patamar de elite.

## 7. Catálogo de Dados

### 7.1 Arquitetura de camadas

```
bronze.imdb_raw               → dado bruto + metadados (ingested_at, source_file)
silver.imdb_clean             → dado limpo, tipado, deduplicado
gold.fato_filmes              → tabela fato com faixas derivadas
gold.dim_genero               → dimensão de gênero (1 linha por filme-gênero)
gold.avg_rating_by_genre      → agregação por gênero (PG1)
gold.avg_rating_by_decade     → agregação por década (PG2)
gold.avg_rating_by_duration   → agregação por faixa de duração (PG3)
gold.top_directors            → ranking de diretores (PG5)
gold.dq_summary               → resumo auditável de qualidade de dados
```

### 7.2 Catálogo — `bronze.imdb_raw`

| Campo | Tipo | Descrição | Domínio |
|-------|------|-----------|---------|
| Poster_Link | string | URL do pôster | URL válida |
| Series_Title | string | Título do filme | texto livre |
| Released_Year | string | Ano bruto (texto) | 4 dígitos |
| Certificate | string | Classificação etária | G, PG, PG-13, R, U, A, UA |
| Runtime | string | Duração bruta ("142 min") | texto |
| Genre | string | Gêneros separados por vírgula | Drama, Crime, ... |
| IMDB_Rating | double | Nota IMDB | 0–10 |
| Overview | string | Sinopse | texto livre |
| Meta_score | double | Nota Metacritic | 0–100 ou nulo |
| Director | string | Diretor | texto livre |
| Star1–Star4 | string | Elenco principal | texto livre |
| No_of_Votes | long | Nº de votos | > 0 |
| Gross | string | Bilheteria bruta ("28,341,469") | texto |
| ingested_at | timestamp | Data/hora da ingestão | ISO 8601 |
| source_file | string | Arquivo de origem | `imdb_top_1000.csv` |

### 7.3 Catálogo — `silver.imdb_clean`

Mesmas colunas da Bronze, com os seguintes ajustes:

- `Released_Year` → **int**
- `Runtime_min` → **int** (nova coluna, derivada de `Runtime`)
- `Meta_score` → **int**
- `Gross` → **double**
- `Poster_Link`, `Overview`, `Star1–Star4` mantidas para rastreabilidade.

### 7.4 Catálogo — `gold.fato_filmes`

| Campo | Tipo | Descrição |
|-------|------|-----------|
| Series_Title | string | Título |
| Director | string | Diretor |
| Certificate | string | Classificação etária |
| Released_Year | int | Ano |
| Runtime_min | int | Duração em minutos |
| Genre | string | Gêneros (multivalorado) |
| IMDB_Rating | float | Nota IMDB |
| Meta_score | int | Metacritic |
| No_of_Votes | long | Nº de votos |
| Gross | double | Bilheteria (USD) |
| periodo | string | `"2000+"` ou `"Antes de 2000"` |
| faixa_duracao | string | `"Curto"` / `"Médio"` / `"Longo"` |

### 7.5 Catálogo — `gold.dim_genero`

| Campo | Tipo | Descrição |
|-------|------|-----------|
| Series_Title | string | Título (FK para `fato_filmes`) |
| IMDB_Rating | float | Nota (desnormalizada para agilidade) |
| Released_Year | int | Ano |
| genero | string | Gênero individual (após `explode`) |

### 7.6 Catálogo — agregações Gold

Cada tabela tem **chave própria + métricas** (não existe coluna genérica `chave`):

| Tabela | Coluna-chave | Métricas |
|--------|--------------|----------|
| `gold.avg_rating_by_genre` | `genero` (string) | `media_nota` (double), `qtd_filmes` (long) |
| `gold.avg_rating_by_decade` | `decada` (long) | `media_nota` (double), `qtd_filmes` (long) |
| `gold.avg_rating_by_duration` | `faixa_duracao` (string) | `media_nota` (double), `media_votos` (double), `qtd_filmes` (long) |
| `gold.top_directors` | `Director` (string) | `media_nota` (double), `qtd_filmes` (long) |

### 7.6b Catálogo — `gold.dq_summary`

| Campo | Tipo | Descrição |
|-------|------|-----------|
| metrica | string | Nome do indicador de qualidade |
| valor | double | Valor medido |
| detalhe | string | Contexto explicativo do indicador |

### 7.7 Linhagem

```
Kaggle CSV
    ↓
bronze.imdb_raw
    ↓
silver.imdb_clean
    ↓
gold.fato_filmes  /  gold.dim_genero
    ↓
agregações gold.*  +  gold.dq_summary
```

O Databricks **Unity Catalog** registra a linhagem automaticamente (aba "Lineage" de cada tabela). Para evidência, consultar o screenshot do catálogo (`docs/screenshots/01_catalog.png` no repositório GitHub).

## 8. Autoavaliação

**Objetivos atingidos:** ✅ Todas as 5 perguntas de negócio foram respondidas com dados persistidos nas 3 camadas do Lakehouse (bronze, silver, gold). ✅ Pipeline reprodutível de ponta a ponta em um único notebook. ✅ Catálogo de dados completo documentado no README + Unity Catalog.

**Dificuldades encontradas:**
1. **Tipagem do `Released_Year`:** o pandas leu como string, exigindo `cast` + `rlike` para filtrar valores não-numéricos sem perder registros válidos.
2. **`Genre` multivalorado:** a média por gênero ficava distorcida quando um filme pertencia a múltiplos gêneros. Resolvido com a `gold.dim_genero` explodida (`explode(split(...))`).
3. **Performance no Free Edition:** agregações feitas em SQL (não em pandas) para aproveitar o motor Spark e respeitar os limites do cluster gratuito.
4. **Outliers bilaterais:** após feedback, refinamos a análise para considerar **2σ bilateral** e **IQR (1.5×)** — descobrimos que os outliers são apenas superiores (clássicos do topo), confirmando que o dataset é curado.

**Trabalhos futuros:**
- Ingestão incremental (Auto Loader) para atualizar o Top 1000 periodicamente.
- Adicionar segunda fonte (ex.: séries de TV ou bilheteria ajustada por inflação).
- Dashboard no Databricks SQL para consumo executivo.
- Testes automatizados de qualidade (DLT expectations / Great Expectations).
- Publicar camadas Gold como *Data Products* no Unity Catalog, com SLAs.

**Aprendizado principal:** entender que **engenharia de dados começa pelo "porquê"**. Sem perguntas claras, o pipeline vira exercício técnico vazio. A definição do objetivo orienta todas as decisões subsequentes — fontes, modelagem, granularidade temporal e tipo de agregação.